In [156]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [170]:
df_new = pd.read_csv('OctNov_CarData.csv')
df_old = pd.read_csv('total_call_data.csv')
df_new.shape

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\2688094094.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old = pd.read_csv('total_call_data.csv')


(331992, 9)

In [171]:
(df_old['Contact Session ID'].nunique())

240592

In [172]:
# Split old data set into before and after Oct 15
# Convert timestamps
df_old['Activity Start Timestamp'] = pd.to_datetime(df_old['Activity Start Timestamp'], format='mixed')
df_new['Activity Start Timestamp'] = pd.to_datetime(df_new['Activity Start Timestamp'], format='mixed')

# Get the first timestamp for each Contact Session ID in df_new
first_timestamps = df_new.groupby('Contact Session ID')['Activity Start Timestamp'].min()

# Create a mapping of Contact Session ID to whether it should go to old dataset
cutoff_date = pd.to_datetime('2025-10-15 23:59:59')
session_to_old = first_timestamps <= cutoff_date

# Split df_new based on the first timestamp of each session
df_new_before = df_new[df_new['Contact Session ID'].map(session_to_old).fillna(False)]
df_new = df_new[~df_new['Contact Session ID'].map(session_to_old).fillna(True)]

# Concatenate the sessions that started before cutoff to df_old
df_old = pd.concat([df_old, df_new_before])

In [162]:
df_new['Queue Name'].value_counts()

Queue Name
Staff Directory English Transfer        8610
Clinic Voicemail Transfer               7608
Front Desk Transfer                     4926
Intake Outdial Queue                    2524
Family                                  1422
Consumer                                1241
Housing                                  852
Staff Directory Spanish Transfer         807
Benefits                                 796
Criminal Records Voicemail Transfer      673
SubSenior Family                         397
SubSenior Consumer                       378
SubSenior Benefits                       313
SubSenior Tenant                         281
Employment                               280
SubSenior Homeowner                      257
SubSenior ADAPT                          257
HIV Voicemail Transfer                   135
Family SP                                118
ADAPT                                    102
Immigration SP                            81
SubSenior Employment                      76

In [163]:
df_new.shape

(243424, 9)

In [164]:
#Filter dataframe
# Steps, check all rows for an ID
# If all the Queue Names for that ID are either na or contain 'Transfer', drop that ID entirely
def filter_transfer_only(df):
    ids_to_drop = []
    for contact_id, group in df.groupby('Contact Session ID'):
        queue_names = group['Queue Name'].dropna().astype(str)
        if all('transfer' in name.lower() or 'intake outdial' in name.lower() for name in queue_names) or queue_names.empty:
            ids_to_drop.append(contact_id)
    filtered_df = df[~df['Contact Session ID'].isin(ids_to_drop)]
    return filtered_df

df_new_filt = filter_transfer_only(df_new)
df_old_filt = filter_transfer_only(df_old)

In [165]:
df_new.shape, df_new_filt.shape, df_old.shape, df_old_filt.shape

((243424, 9), (30820, 9), (3310855, 9), (421624, 9))

In [166]:
df_new_filt = df_new.copy()
df_old_filt =df_old.copy()

In [167]:
ids = df_new[df_new['Activity Name'].str.contains('SuburbsOrCity', na = False)]['Contact Session ID'].unique()

df_new[df_new['Contact Session ID'] == ids[0]]

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
50613,da6e0515-1021-4c13-942c-124768534bcd,Main Number Telephony EP,NaN,NaN,2025-10-16 08:03:15,NaN,NaN,NaN,8
50614,da6e0515-1021-4c13-942c-124768534bcd,NaN,LACMain,NaN,2025-10-16 08:03:16,NaN,NaN,NaN,8
50615,da6e0515-1021-4c13-942c-124768534bcd,NaN,LegalMenu,NaN,2025-10-16 08:03:16,NaN,NaN,NaN,8
50616,da6e0515-1021-4c13-942c-124768534bcd,Legal Menu Telephony EP,NaN,LegalMenu1,2025-10-16 08:03:16,NaN,NaN,NaN,8
50617,da6e0515-1021-4c13-942c-124768534bcd,Legal Menu Telephony EP,LegalMenu,NaN,2025-10-16 08:03:16,NaN,NaN,NaN,8
50619,da6e0515-1021-4c13-942c-124768534bcd,Legal Menu Telephony EP,NaN,LegalMenu2,2025-10-16 08:03:18,NaN,NaN,NaN,8
50628,da6e0515-1021-4c13-942c-124768534bcd,NaN,SeniorsMenu,NaN,2025-10-16 08:03:24,NaN,NaN,NaN,8
50629,da6e0515-1021-4c13-942c-124768534bcd,Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-10-16 08:03:24,NaN,NaN,NaN,8
50639,da6e0515-1021-4c13-942c-124768534bcd,Seniors Menu Telephony EP,NaN,SuburbsOrCityMenu,2025-10-16 08:03:27,NaN,NaN,NaN,8
50649,da6e0515-1021-4c13-942c-124768534bcd,NaN,Queues,NaN,2025-10-16 08:03:44,NaN,NaN,NaN,8


In [168]:
# First Step: Check if proportion of senior menu queues is the same in both datasets
# Filter out all queues that contain transfer or intake outdial

ids_senior_new = df_new[df_new['Activity Name'].str.contains('SuburbsOrCity', na=False)]['Contact Session ID'].unique()
ids_senior_old = df_old[df_old['Activity Name'].str.contains('SuburbsOrCity', na=False)]['Contact Session ID'].unique()


prop_senior_new = len(ids_senior_new) / df_new['Contact Session ID'].nunique()
prop_senior_old = len(ids_senior_old) / df_old['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")
print(f"Total CitySenior new IDs: {len(ids_senior_new)}, total new IDS {df_new['Contact Session ID'].nunique()},  CitySenior old IDs: {len(ids_senior_old)}, total old IDs {df_old['Contact Session ID'].nunique()}")

prop new: 7.1297 %, prop old: 11.3040 %
Total CitySenior new IDs: 1359, total new IDS 19061,  CitySenior old IDs: 27960, total old IDs 247346


In [169]:
ids_senior_new = df_new[df_new['Activity Name'].str.contains('GetLoggedInSubSenior', na=False)]['Contact Session ID'].unique()
ids_senior_old = df_old[df_old['Activity Name'].str.contains('GetLoggedInSubSenior', na=False)]['Contact Session ID'].unique()


prop_senior_new = len(ids_senior_new) / df_new['Contact Session ID'].nunique()
prop_senior_old = len(ids_senior_old) / df_old['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")
print(f"Total SubSenior new IDs: {len(ids_senior_new)}, total new IDS {df_new['Contact Session ID'].nunique()},  SubSenior old IDs: {len(ids_senior_old)}, total old IDs {df_old['Contact Session ID'].nunique()}")

prop new: 1.7313 %, prop old: 2.4941 %
Total SubSenior new IDs: 330, total new IDS 19061,  SubSenior old IDs: 6169, total old IDs 247346


After some explatory analysis, (which can and should be double checked) it appears that in Activity Name, before in all instances before the new implementation SeniorsMenu shows up prior to LegalMenu1, while in instances after the implementation SeniorsMenu now shows up after LegalMenu1(and 2). 

Next, I learned that a successful Senior would be determined by GetLoggedInSubSenior... in the Activity Name. It appears that this Activity Name was essentially a confirmation of what ever SubSeniorMenu they chose. 

Next Steps: The goal now is to find the amonut of time it takes for a customer who gets to Senior menu to make a call. This can be found in a handful of ways. 

Method 1: Finding total length of calls for all these people. This is easiest method, but leads to the length of time waiitng in queue or being on call with an agent to have a potentially large impact.

Method 2: Find length of call from start until we get to PreQueueMessage or ClosedQueue. This takes away any waiting or speaking bias, but it potentially will leave out some people if they never end up in a queue. I am not sure how many people(if any) will be left out by this determination. A possible solution is simply to take time of full call for those who never get to a queue or closedqueue. 

Method 3: Taking time from start of call to LegalMenu1. I am not sure, but even though SeniorsMenu shows up after LegalMenu1 in the new dataset there is a chance that that is a placeholder and does not actually take up anytime. Will have to check how times are recorded for both before and after implemenation and see if this idea seems ot make any sense or not. 

In [174]:
[x for x in list(df_new['Activity Name'].unique()[1:]) if 'GetLoggedIn' in x]

['GetLoggedInConsumerAgents',
 'GetLoggedInFamilyAgents',
 'GetLoggedInBenefitsAgents',
 'GetLoggedInBenefitsSPAgents',
 'GetLoggedInSubSeniorConsumerSPAgents',
 'GetLoggedInHousingAgents',
 'GetLoggedInConsumerSPAgents',
 'GetLoggedInSubSeniorHomeownerAgents',
 'GetLoggedInSubSeniorBenefitsAgents',
 'GetLoggedInFamilySPAgents',
 'GetLoggedInEmploymentAgents',
 'GetLoggedInSubSeniorEmploymentAgents',
 'GetLoggedInSubSeniorFamilyAgents',
 'GetLoggedInSubSeniorADAPTAgents',
 'GetLoggedInADAPTAgents',
 'GetLoggedInSubSeniorConsumerAgents',
 'GetLoggedInSubSeniorTenantAgents',
 'GetLoggedInImmigrationAgents',
 'GetLoggedInSubSeniorBenefitsSPAgents',
 'GetLoggedInEmploymentSPAgents',
 'GetLoggedInImmigrationSPAgents',
 'GetLoggedInADAPTSPAgents',
 'GetLoggedInSubSeniorFamilySPAgents',
 'GetLoggedInSubSeniorEmploymentSPAgents',
 'GetLoggedInSubSeniorADAPTSPAgents',
 'GetLoggedInEducationAgents',
 'GetLoggedInHousingSPAgents',
 'GetLoggedInEducationSPAgents']

In [93]:
list(df_new['Activity Name'].unique()[1:])

['LanguageSelectionMenu',
 'ClosedMenu',
 'DisconnectContact1',
 'AddressFaxHoursMenu',
 'CallbackRetry',
 'FarmworkerMainMenu',
 'StaffDirectoryEnglishTransfer',
 'MainMenu',
 'AppointmentMenu',
 'LegalMenu1',
 'LegalMenu2',
 'FamilyMenu',
 'HousingMenu',
 'SeniorsMenu',
 'GetLoggedInConsumerAgents',
 'DivorceOrParentingMenu',
 'ClosedQueueMenu',
 'BenefitsMenu',
 'SimpleDivorceMenu',
 'GetLoggedInFamilyAgents',
 'GetLoggedInBenefitsAgents',
 'EmploymentMenu',
 'GetLoggedInBenefitsSPAgents',
 'ClinicVoicemailTransfer',
 'SuburbsOrCityMenu',
 'GetLoggedInSubSeniorConsumerSPAgents',
 'OtherLegalMenu',
 'OtherLegalOtherMenu',
 'ComplimentOrComplaintMenu',
 'LegalServerScreenPop',
 'PreTenantMenu',
 'TenantMenu',
 'IntakePreQueueMessage1',
 'GetLoggedInHousingAgents',
 'ConsumerQueue',
 'PreQueueMessage2',
 'HousingQueue',
 'PlayMOH300s',
 'ReadANI',
 'CCB',
 'PlayCCBConfirmation',
 'GetLoggedInConsumerSPAgents',
 'CollectCallbackNumber',
 'ConsumerSPQueue',
 'ConfirmCallbackNumber',
 'Qu

In [62]:
temp = df_new_filt[df_new_filt['Contact Session ID'].isin(ids_senior_new)]
temp['Queue Name'].value_counts()

Queue Name
SubSenior Family         397
SubSenior Consumer       378
SubSenior Benefits       313
SubSenior Tenant         281
SubSenior ADAPT          257
SubSenior Homeowner      257
SubSenior Employment      76
SubSenior Family SP       52
SubSenior Consumer SP     49
SubSenior Benefits SP     23
SubSenior ADAPT SP         3
Name: count, dtype: int64

In [81]:
df_old['Activity Name'].value_counts()[:60]

Activity Name
LanguageSelectionMenu               265758
MainMenu                            184476
SeniorsMenu                         124718
LegalMenu2                          109995
LegalMenu1                           95509
ClosedQueueMenu                      87246
OtherLegalMenu                       31689
SeniorsConfirmationMenu              30067
SuburbsOrCityMenu                    28110
StaffDirectoryEnglishTransfer        27869
FamilyMenu                           27074
PlayMOH300s                          25488
ClinicVoicemailTransfer              25459
QueueMenu1                           23947
ClosedMenu                           23670
HousingMenu                          23061
SeniorsADAPTMenu                     21522
DisconnectContact1                   20549
SetCallerID                          18550
TenantMenu                           17178
PreTenantMenu                        15979
DivorceOrParentingMenu               14956
AppointmentMenu                      138

In [80]:
df_new['Activity Name'].value_counts()[:60]

Activity Name
LanguageSelectionMenu               19608
MainMenu                            13385
LegalMenu2                           9768
LegalMenu1                           8703
SeniorsMenu                          5757
ClosedQueueMenu                      5279
FamilyMenu                           2447
OtherLegalMenu                       2293
PlayMOH300s                          2183
StaffDirectoryEnglishTransfer        2153
HousingMenu                          2129
QueueMenu1                           2037
ClinicVoicemailTransfer              1922
ClosedMenu                           1785
SuburbsOrCityMenu                    1403
TenantMenu                           1349
PreTenantMenu                        1323
DivorceOrParentingMenu               1298
FarmworkerMainMenu                   1293
OtherLegalOtherMenu                  1163
SetCallerID                          1018
AppointmentMenu                       968
HelpWithLegalorOtherReasonMenu        856
LegalServerScreenPop

In [66]:
df_new.loc[df_new['Contact Session ID'] == '2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
50447,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50448,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,LACMain,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50449,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-10-16 08:00:06,NaN,NaN,NaN,8
50450,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,LACMain,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50467,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,MainMenu,2025-10-16 08:00:17,NaN,NaN,NaN,8
50471,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,LegalMenu,NaN,2025-10-16 08:00:24,NaN,NaN,NaN,8
50472,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Legal Menu Telephony EP,NaN,LegalMenu1,2025-10-16 08:00:24,NaN,NaN,NaN,8
50510,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Legal Menu Telephony EP,NaN,LegalMenu2,2025-10-16 08:01:00,NaN,NaN,NaN,8
50544,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,SeniorsMenu,NaN,2025-10-16 08:02:07,NaN,NaN,NaN,8
50545,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-10-16 08:02:07,NaN,NaN,NaN,8


In [70]:
df_old.loc[df_old['Contact Session ID'] == '001a3748-8d50-4550-8461-33547983deb0']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
0,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
1,001a3748-8d50-4550-8461-33547983deb0,NaN,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
2,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-14 12:39:32,NaN,NaN,NaN,12
3,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
4,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,MainMenu,2025-01-14 12:39:45,NaN,NaN,NaN,12
5,001a3748-8d50-4550-8461-33547983deb0,NaN,PreLegalMenuSeniorsMenu,NaN,2025-01-14 12:40:09,NaN,NaN,NaN,12
6,001a3748-8d50-4550-8461-33547983deb0,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-01-14 12:40:09,NaN,NaN,NaN,12
7,001a3748-8d50-4550-8461-33547983deb0,NaN,LegalMenu,NaN,2025-01-14 12:40:18,NaN,NaN,NaN,12
8,001a3748-8d50-4550-8461-33547983deb0,Legal Menu Telephony EP,NaN,LegalMenu1,2025-01-14 12:40:18,NaN,NaN,NaN,12
9,001a3748-8d50-4550-8461-33547983deb0,Legal Menu Telephony EP,NaN,LegalMenu2,2025-01-14 12:40:54,NaN,NaN,NaN,12
